In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder,StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, accuracy_score, f1_score, confusion_matrix, precision_score, recall_score, classification_report,roc_auc_score,roc_curve,RocCurveDisplay
import os
from sklearn.compose import ColumnTransformer,make_column_selector
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from tqdm import tqdm
os.chdir('/home/pgcp-ai/MachineLearning/Datasets/')

In [2]:
hr = pd.read_csv("HR_comma_sep.csv")
hr

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.10,0.77,6,247,4,0,1,0,sales,low
3,0.92,0.85,5,259,5,0,1,0,sales,low
4,0.89,1.00,5,224,5,0,1,0,sales,low
...,...,...,...,...,...,...,...,...,...,...
14990,0.40,0.57,2,151,3,0,1,0,support,low
14991,0.37,0.48,2,160,3,0,1,0,support,low
14992,0.37,0.53,2,143,3,0,1,0,support,low
14993,0.11,0.96,6,280,4,0,1,0,support,low


In [3]:
X, y = hr.drop("left", axis = 1), hr["left"]

In [4]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,stratify=y,random_state=26)

In [5]:
ohe = OneHotEncoder(sparse_output=False,drop='first',handle_unknown='ignore').set_output(transform='pandas')
ss = StandardScaler().set_output(transform='pandas')

In [6]:
transformer = ColumnTransformer(transformers = [('OHE',ohe,make_column_selector(dtype_include=object))],
                               remainder='passthrough',
                               verbose_feature_names_out=False,
                               )

In [7]:
X_train_ohe = transformer.fit_transform(X_train)
X_test_ohe = transformer.transform(X_test)

In [8]:
X_train_scaled = ss.fit_transform(X_train_ohe)
X_test_scaled = ss.transform(X_test_ohe)

In [9]:
svm = SVC(kernel='linear')
svm.fit(X_train_scaled,y_train)
y_pred = svm.predict(X_test_scaled)
accuracy_score(y_test,y_pred)

0.7781729273171816

In [10]:
Cs = np.linspace(0.001,5,20)
scores = []
for c in tqdm(Cs):
    svm = SVC(kernel='linear',C=c,probability=True,random_state=26)
    svm.fit(X_train_scaled,y_train)
    y_pred = svm.predict(X_test_scaled)
    acc = accuracy_score(y_test,y_pred)
    y_pred_prob = svm.predict_proba(X_test_scaled)
    log_loss_score = log_loss(y_test,y_pred_prob)
    scores.append([c,acc,log_loss_score])

100%|███████████████████████████████████████████| 20/20 [14:42<00:00, 44.14s/it]


In [11]:
df_scores = pd.DataFrame(scores,columns=['C','Accuracy Score','Log Loss'])
df_scores.sort_values('Log Loss',ascending=True)

,C,Accuracy Score,Log Loss
9,2.368947,0.778173,0.437984
14,3.684474,0.778173,0.437994
15,3.947579,0.778173,0.437995
8,2.105842,0.778173,0.437995
11,2.895158,0.778173,0.438002
18,4.736895,0.778173,0.438006
4,1.053421,0.778173,0.438008
17,4.473789,0.778173,0.438019
12,3.158263,0.778173,0.438020
6,1.579632,0.778173,0.438020


### To use `predict_proba` in SVC we have to first put `probability=True` and re-assign `random_state=26` in SVC Constructor to enable `predict_proba` outputs and avoid errors

In [12]:
svm = SVC(C=0.790316,kernel='linear',probability=True,random_state=26)
svm.fit(X_train_scaled,y_train)
y_pred_prob = svm.predict_proba(X_test_scaled)
log_loss(y_test,y_pred_prob)

0.43802887627713

### Radial Kernel

In [16]:
Cs = np.linspace(0.001,5,10)
Gs = np.linspace(0.001, 5, 10)
scores = []
for c in tqdm(Cs):
    for g in Gs: 
        svm = SVC(kernel='rbf',C=c,probability=True,gamma = g,random_state=26)
        svm.fit(X_train_scaled,y_train)
        y_pred = svm.predict(X_test_scaled)
        acc = accuracy_score(y_test,y_pred)
        y_pred_prob = svm.predict_proba(X_test_scaled)
        log_loss_score = log_loss(y_test,y_pred_prob)
        scores.append([c,g,acc,log_loss_score])

100%|████████████████████████████████████████| 10/10 [1:00:14<00:00, 361.49s/it]


In [17]:
df_scores = pd.DataFrame(scores, columns = ['C', 'G', 'Accuracy Score', 'Log Loss'])

In [18]:
df_scores.sort_values(['Log Loss', 'Accuracy Score'], ascending = [True, False])

,C,G,Accuracy Score,Log Loss
33,1.667333,1.667333,0.977106,0.085636
43,2.222778,1.667333,0.976884,0.085688
42,2.222778,1.111889,0.976884,0.086147
53,2.778222,1.667333,0.976661,0.086291
52,2.778222,1.111889,0.976439,0.086528
...,...,...,...,...
40,2.222778,0.001000,0.803512,0.399037
10,0.556444,0.001000,0.762169,0.406913
30,1.667333,0.001000,0.808180,0.406953
20,1.111889,0.001000,0.766615,0.406988
